# 10 — Tầng 11: Dashboard và Agent Layer

**Vị trí trong pipeline** (`../method/AGENT1_TONG_HOP.md` Tầng 11):
```
04_calibration → 05_monte_carlo → 06_stress_test → 07_evaluation
              → 08_survival → 09_collection → [10_dashboard]
```

Plan Tầng 11 nói rõ: *"Dashboard không nên chỉ show model metric. Cần show quyết định."*
Và: *"LLM/agent chỉ nên dùng để giải thích và soạn hành động thu hồi nợ. Không để LLM tự
tính forecast."*

Notebook này gom toàn bộ artifact của 9 notebook trước thành **một màn hình cho CFO** và
**một hợp đồng dữ liệu cho agent**, kèm những gì bắt buộc phải hiển thị:

| Thành phần bắt buộc (plan) | Nguồn |
|---|---|
| Cash-in P10/P50/P90 theo ngày/tuần | notebook 05 |
| Shortfall risk so với ngưỡng kế hoạch | notebook 05 |
| Top invoices/customers tạo rủi ro | notebook 05, 09 |
| Reason codes | notebook 09 |
| Forecast source (baseline / LightGBM / survival / stress) | notebook 07, 08 |
| Confidence regime (normal / drift / crisis) | notebook 06 |

**Đầu ra chính:** `reports/dashboard.html` — một file HTML tự chứa, mở được bằng trình duyệt,
không cần server, không cần internet.

## 📋 Decision Log (D59 → D63)

---

### D59 · Dashboard phải hiển thị số nào khi hai model chênh nhau $213 triệu

Notebook 08 để lại một tình huống thật: LightGBM nói $308M về trong 30 ngày, Survival B nói
$95M. Dashboard **không được** chọn im lặng một con số. Quy tắc hiển thị:

1. **Số chính** = số của regime hiện tại (notebook 06). Đang `crisis` → lấy nhánh thận trọng.
2. **Số đối chiếu** luôn hiện ngay cạnh, kèm nhãn giả định của nó.
3. Mỗi con số mang **nhãn nguồn** và **nhãn độ tin cậy**.

Một dashboard giấu đi sự bất định giữa các model thì tệ hơn không có dashboard, vì nó tạo
cảm giác chắc chắn mà bằng chứng không có.

---

### D60 · Ba mức độ tin cậy, gắn vào từng con số

| Mức | Điều kiện | Cách hiển thị |
|---|---|---|
| `đã kiểm chứng` | có backtest ngoài mẫu trên pool tương tự | số thường |
| `thận trọng` | có backtest nhưng pool khác với hiện tại | kèm cảnh báo |
| `chưa kiểm chứng` | không có bằng chứng ngoài mẫu | in nghiêng + biểu tượng cảnh báo |

Con số $312M cash-in 30 ngày thuộc nhóm thứ ba (notebook 08 §6), và dashboard phải nói ra
điều đó thay vì để người đọc tự đoán.

---

### D61 · Agent layer: hợp đồng, không phải lời khuyên

LLM **được** làm: giải thích reason codes bằng ngôn ngữ tự nhiên, soạn thư nhắc nợ, tóm tắt
thay đổi tuần này so với tuần trước, trả lời câu hỏi dựa trên context pack.

LLM **không được** làm: tính hay sửa forecast, tự quyết định thứ tự gọi, gửi thư mà không có
người duyệt, suy diễn con số không có trong context pack.

Cách bảo đảm về mặt kỹ thuật: agent chỉ nhận **context pack JSON** đã tính sẵn, mọi con số
trong thư đều là trường của JSON đó. §5 sinh context pack và một bản nháp thư **bằng template
tất định** — để chứng minh rằng phần sinh chữ không cần đến LLM để đúng số.

---

### D62 · Dashboard tự chứa, không phụ thuộc hạ tầng

Xuất HTML một file, biểu đồ nhúng base64. Lý do: hồ sơ dự thi phải mở được trên máy giám
khảo, không thể yêu cầu cài thư viện hay chạy server. Đánh đổi: không tương tác được — chấp
nhận, vì bản prototype cần **đúng và mở được** hơn là đẹp.

---

### D63 · Panel hạn chế là thành phần bắt buộc, không phải phần phụ

Dashboard có một panel cố định liệt kê hạn chế đã biết. Đây là yêu cầu thiết kế: người đọc
phải thấy hạn chế **cùng lúc** với con số, không phải ở trang cuối báo cáo mà không ai đọc.

In [1]:
import base64
import io
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)
pd.set_option("display.float_format", "{:,.3f}".format)

BASE = Path("D:/CURRICULUM/Semester VI/Finance Project")
PROC = BASE / "data" / "processed"
REPORTS = BASE / "reports"
REPORTS.mkdir(exist_ok=True)

OUT_HTML = REPORTS / "dashboard.html"
OUT_INVOICE_VIEW = PROC / "dashboard_invoice_view.csv"
OUT_KPI = PROC / "dashboard_kpi.csv"
OUT_CONTEXT = REPORTS / "agent_context_pack.json"
OUT_DRAFTS = REPORTS / "dunning_drafts.md"

C_SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#4a3aa7"]
C_GOOD, C_WARN, C_SERIOUS, C_CRIT = "#0ca30c", "#fab219", "#ec835a", "#d03b3b"
C_INK, C_INK2, C_GRID = "#0b0b0b", "#52514e", "#d8d7d2"
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb",
    "axes.edgecolor": C_GRID, "axes.labelcolor": C_INK2, "axes.titlecolor": C_INK,
    "xtick.color": C_INK2, "ytick.color": C_INK2, "text.color": C_INK,
    "grid.color": C_GRID, "grid.linewidth": 0.8, "axes.grid": True,
    "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 10, "axes.titlesize": 11, "legend.frameon": False,
})
print("Config OK")

Config OK


## 1. Gom artifact của cả pipeline

In [2]:
def load(name, **kw):
    p = PROC / name
    return pd.read_csv(p, **kw) if p.exists() else None


daily = load("mc_daily_cash_forecast.csv", parse_dates=["date"])
weekly = load("mc_weekly_cash_forecast.csv", parse_dates=["week_end"])
horizon = load("mc_horizon_summary.csv")
shortfall = load("mc_shortfall_report.csv")
stress = load("stress_scenario_summary.csv")
regime = load("regime_status.csv")
psi = load("psi_train_vs_live.csv")
surv_live = load("survival_live_forecast.csv")
money = load("survival_money_check.csv")
worklist = load("collection_worklist.csv", parse_dates=["due_date"])
scores = load("collection_live_scores.csv")
inv_risk = load("mc_invoice_risk.csv", parse_dates=["due_date"])
cust_risk = load("mc_customer_risk.csv")
gate = load("eval_gate_decision.csv")
cash_metrics = load("eval_cash_metrics.csv")
inv_metrics = load("eval_invoice_metrics.csv")
quality = load("data_quality_report.csv")
cqr_cov = load("cqr_coverage_report.csv")

AS_OF = pd.Timestamp(daily["date"].min()) - pd.Timedelta(days=1)
REGIME = regime["regime"].iloc[0]
GATE_DECISION = gate["final_decision"].iloc[0]
loaded = {k: (v is not None) for k, v in {
    "mc_daily": daily, "mc_weekly": weekly, "stress": stress, "regime": regime,
    "survival_live": surv_live, "worklist": worklist, "gate": gate, "psi": psi}.items()}
print(f"As-of: {AS_OF.date()} | regime: {REGIME.upper()} | gate Tầng 9: {GATE_DECISION}")
print("Artifact nạp được:", {k: ("có" if v else "THIẾU") for k, v in loaded.items()})
missing = [k for k, v in loaded.items() if not v]
if missing:
    raise FileNotFoundError(f"Thiếu artifact bắt buộc: {missing} — chạy lại notebook tương ứng.")

As-of: 2020-05-22 | regime: CRISIS | gate Tầng 9: CONDITIONAL
Artifact nạp được: {'mc_daily': 'có', 'mc_weekly': 'có', 'stress': 'có', 'regime': 'có', 'survival_live': 'có', 'worklist': 'có', 'gate': 'có', 'psi': 'có'}


## 2. Khối KPI — và quy tắc chọn số chính khi hai model bất đồng (D59, D60)

In [3]:
open_amount = float(inv_risk["amount"].sum())
h30 = horizon[horizon["horizon_days"] == 30].iloc[0]

surv_b_30 = float((surv_live["amount"] * surv_live["b_prob_paid_30d"]).sum())
surv_a_30 = float((surv_live["amount"] * surv_live["a_prob_paid_30d"]).sum())
lgbm_30 = float((surv_live["amount"] * surv_live["lgbm_prob_paid_30d"]).sum())

if REGIME == "crisis":
    headline_30, headline_src, headline_conf = surv_b_30, "Survival B (chế độ khủng hoảng)", "thận trọng"
    alt_30, alt_src = lgbm_30, "LightGBM + CQR + Monte Carlo (giả định hồi phục)"
else:
    headline_30, headline_src, headline_conf = float(h30["mean"]), "LightGBM + CQR + Monte Carlo", "thận trọng"
    alt_30, alt_src = surv_b_30, "Survival B (chế độ khủng hoảng)"

kpi = pd.DataFrame([
    {"kpi": "Dư nợ AR đang mở", "value": open_amount, "unit": "USD",
     "source": "invoices_ar_clean", "confidence": "đã kiểm chứng"},
    {"kpi": "Số hoá đơn đang mở", "value": float(len(inv_risk)), "unit": "hoá đơn",
     "source": "invoices_ar_clean", "confidence": "đã kiểm chứng"},
    {"kpi": "Tiền về 30 ngày — SỐ CHÍNH", "value": headline_30, "unit": "USD",
     "source": headline_src, "confidence": headline_conf},
    {"kpi": "Tiền về 30 ngày — số đối chiếu", "value": alt_30, "unit": "USD",
     "source": alt_src, "confidence": "chưa kiểm chứng"},
    {"kpi": "Dải P10–P90 30 ngày (Monte Carlo)", "value": float(h30["p90"] - h30["p10"]), "unit": "USD",
     "source": "notebook 05", "confidence": "chưa kiểm chứng"},
    {"kpi": "Tiền rủi ro 30 ngày (không kịp về)", "value": float(inv_risk["risk_amount_30d"].sum()),
     "unit": "USD", "source": "notebook 05", "confidence": "thận trọng"},
    {"kpi": "PSI tối đa train vs live", "value": float(psi["psi"].max()), "unit": "chỉ số",
     "source": "notebook 06", "confidence": "đã kiểm chứng"},
    {"kpi": "Coverage dải hoá đơn trên test", "value": float(
        cqr_cov.query("scope=='split' and key=='test'")["cov_after"].iloc[0]), "unit": "tỷ lệ",
     "source": "notebook 04", "confidence": "đã kiểm chứng"},
])
print("KPI dashboard")
print(kpi.to_string(index=False))
print()
print(f"Regime = {REGIME.upper()} → số chính lấy từ: {headline_src}")
print(f"  Số chính     : ${headline_30:,.0f}  ({headline_30/open_amount:.1%} dư nợ)")
print(f"  Số đối chiếu : ${alt_30:,.0f}  ({alt_30/open_amount:.1%} dư nợ)")
print(f"  Chênh lệch   : ${abs(alt_30 - headline_30):,.0f}")
print()
print("Quy tắc D59: cả hai con số cùng xuất hiện trên màn hình, kèm giả định của mỗi bên.")

KPI dashboard
                               kpi           value    unit                                           source      confidence
                  Dư nợ AR đang mở 312,538,768.520     USD                                invoices_ar_clean   đã kiểm chứng
                Số hoá đơn đang mở       9,681.000 hoá đơn                                invoices_ar_clean   đã kiểm chứng
        Tiền về 30 ngày — SỐ CHÍNH  95,427,069.561     USD                  Survival B (chế độ khủng hoảng)      thận trọng
    Tiền về 30 ngày — số đối chiếu 307,944,641.972     USD LightGBM + CQR + Monte Carlo (giả định hồi phục) chưa kiểm chứng
 Dải P10–P90 30 ngày (Monte Carlo)  37,889,242.043     USD                                      notebook 05 chưa kiểm chứng
Tiền rủi ro 30 ngày (không kịp về)  14,369,659.461     USD                                      notebook 05      thận trọng
          PSI tối đa train vs live           6.229  chỉ số                                      notebook 06   đã kiểm 

## 3. Bảng hoá đơn cho dashboard — reason code, nguồn, độ tin cậy

In [4]:
view = inv_risk[["invoice_id", "customer_id", "name_customer", "amount", "due_date",
                 "aging_bucket_at_T", "cal_q10", "cal_q50", "cal_q90",
                 "prob_paid_30d", "risk_amount_30d", "expected_days"]].copy()
view = view.rename(columns={"prob_paid_30d": "p_paid_30d_mc"})

if scores is not None:
    view = view.merge(scores[["invoice_id", "p_paid_7d", "p_paid_30d"]]
                      .rename(columns={"p_paid_30d": "p_paid_30d_clf"}),
                      on="invoice_id", how="left")
view = view.merge(surv_live[["invoice_id", "b_prob_paid_30d", "b_median_remaining_days"]],
                  on="invoice_id", how="left")

view["spread_pp"] = (view["p_paid_30d_mc"] - view["b_prob_paid_30d"]).abs() * 100
view["confidence"] = np.where(view["spread_pp"] > 40, "thấp — hai model bất đồng mạnh",
                              np.where(view["spread_pp"] > 15, "trung bình", "cao"))
view["forecast_source"] = f"LightGBM+CQR+MC | Survival B | regime={REGIME}"

if worklist is not None:
    view = view.merge(worklist[["invoice_id", "priority", "reason_codes"]], on="invoice_id", how="left")
view["in_worklist"] = view["priority"].notna() if "priority" in view.columns else False

print(f"Bảng hoá đơn dashboard: {len(view):,} dòng")
print(view["confidence"].value_counts().to_string())
print()
print("Mức bất đồng giữa hai model theo nhóm aging (điểm phần trăm):")
print(view.groupby("aging_bucket_at_T")["spread_pp"].agg(["count", "mean", "max"]).to_string())
print()
print("Đọc: nhóm quá hạn lâu là nơi hai model bất đồng nhất — cũng chính là nơi tiền đang nằm.")
print("Dashboard gắn nhãn `độ tin cậy thấp` cho đúng những dòng đó thay vì hiển thị một con số phẳng.")

Bảng hoá đơn dashboard: 9,681 dòng
confidence
thấp — hai model bất đồng mạnh    7959
cao                                910
trung bình                         812

Mức bất đồng giữa hai model theo nhóm aging (điểm phần trăm):
                   count   mean    max
aging_bucket_at_T                     
1_30                3567 81.780 98.273
31_60               4427 60.859 94.404
61_90               1187 57.659 96.405
not_due              500 70.096 99.460

Đọc: nhóm quá hạn lâu là nơi hai model bất đồng nhất — cũng chính là nơi tiền đang nằm.
Dashboard gắn nhãn `độ tin cậy thấp` cho đúng những dòng đó thay vì hiển thị một con số phẳng.


## 4. Vẽ biểu đồ và nhúng vào HTML (D62)

In [5]:
def fig_to_b64(fig, dpi=110):
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=dpi, bbox_inches="tight")
    plt.close(fig)
    return base64.b64encode(buf.getvalue()).decode("ascii")


# ── (1) Fan chart dòng tiền + hai kịch bản model ─────────────────────────
fig, ax = plt.subplots(figsize=(9.5, 4.2))
ax.fill_between(daily["date"], daily["cum_p10"] / 1e6, daily["cum_p90"] / 1e6,
                color=C_SERIES[0], alpha=0.16, label="Monte Carlo P10–P90")
ax.plot(daily["date"], daily["cum_p50"] / 1e6, color=C_SERIES[0], lw=2,
        label="Monte Carlo P50 (giả định hồi phục)")
ax.plot(daily["date"], daily["plan_cum"] / 1e6, color=C_CRIT, ls="--", lw=1.5,
        label="Ngưỡng kế hoạch (proxy)")
d30 = daily["date"].iloc[29]
ax.scatter([d30], [surv_b_30 / 1e6], color=C_SERIES[2], s=70, zorder=5,
           label="Survival B tại 30 ngày (chế độ khủng hoảng)")
ax.annotate(f"${surv_b_30/1e6:,.0f}M", (d30, surv_b_30 / 1e6), textcoords="offset points",
            xytext=(8, -4), color=C_SERIES[2], fontsize=9, fontweight="bold")
ax.annotate(f"${daily['cum_p50'].iloc[29]/1e6:,.0f}M", (d30, daily["cum_p50"].iloc[29] / 1e6),
            textcoords="offset points", xytext=(8, 4), color=C_SERIES[0], fontsize=9, fontweight="bold")
ax.set_ylabel("Triệu USD"); ax.set_title("Cash-in tích luỹ từ AR đang mở — hai kịch bản chế độ")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:,.0f}"))
ax.legend(fontsize=8, loc="lower right"); ax.grid(axis="x", visible=False)
img_cash = fig_to_b64(fig)

# ── (2) Stress scenarios ─────────────────────────────────────────────────
sim = stress[stress["kind"] != "policy_haircut"].sort_values("cash_30d_p50")
fig, ax = plt.subplots(figsize=(9.5, 4.2))
y = np.arange(len(sim))
colors = [C_SERIES[0] if s == "S0" else (C_CRIT if d < -0.3 else C_SERIOUS)
          for s, d in zip(sim["scenario_id"], sim["delta_30d_vs_base_pct"].fillna(0))]
ax.barh(y, sim["cash_30d_p50"] / 1e6, color=colors, height=0.6)
ax.set_yticks(y)
ax.set_yticklabels([f"{a} · {b[:38]}" for a, b in zip(sim["scenario_id"], sim["scenario"])], fontsize=8)
ax.set_xlabel("Cash-in 30 ngày, P50 (triệu USD)")
ax.set_title("Sức chịu đựng theo kịch bản stress (Tầng 8)")
ax.grid(axis="y", visible=False)
img_stress = fig_to_b64(fig)

# ── (3) Top khách hàng theo tiền rủi ro ──────────────────────────────────
top_c = cust_risk.nlargest(10, "risk_amount_30d").iloc[::-1]
fig, ax = plt.subplots(figsize=(9.5, 4.2))
y = np.arange(len(top_c))
ax.barh(y, top_c["risk_amount_30d"] / 1e6, color=C_SERIES[1], height=0.62)
ax.set_yticks(y)
ax.set_yticklabels([f"{n[:26]} ({int(i)} hđ)" for n, i in
                    zip(top_c["name_customer"].astype(str), top_c["invoices"])], fontsize=8)
ax.set_xlabel("Tiền có nguy cơ không về trong 30 ngày (triệu USD)")
ax.set_title("Top 10 khách hàng theo tiền rủi ro")
ax.grid(axis="y", visible=False)
img_cust = fig_to_b64(fig)

print("Đã dựng 3 biểu đồ và mã hoá base64 để nhúng vào HTML.")
print(f"  kích thước ảnh: {len(img_cash)//1024}KB, {len(img_stress)//1024}KB, {len(img_cust)//1024}KB")

Đã dựng 3 biểu đồ và mã hoá base64 để nhúng vào HTML.
  kích thước ảnh: 66KB, 70KB, 51KB


## 5. Agent layer — context pack và bản nháp tất định (D61)

In [6]:
context_pack = {
    "as_of": str(AS_OF.date()),
    "regime": REGIME,
    "regime_actions": regime["required_actions"].iloc[0].split(" | "),
    "gate_decision": GATE_DECISION,
    "portfolio": {
        "open_amount_usd": round(open_amount, 2),
        "open_invoices": int(len(inv_risk)),
        "cash_30d_headline_usd": round(headline_30, 2),
        "cash_30d_headline_source": headline_src,
        "cash_30d_alternative_usd": round(alt_30, 2),
        "cash_30d_alternative_source": alt_src,
        "risk_amount_30d_usd": round(float(inv_risk["risk_amount_30d"].sum()), 2),
    },
    "evidence": {
        "psi_max": round(float(psi["psi"].max()), 3),
        "psi_top_feature": psi.iloc[0]["feature"],
        "changepoint": regime["recent_breakpoints"].iloc[0],
        "interval_coverage_test": round(float(
            cqr_cov.query("scope=='split' and key=='test'")["cov_after"].iloc[0]), 3),
        "post_shock_actual_collection_14d": round(float(
            money.query("window=='test_post' and horizon_days==14")["thực tế"].iloc[0]), 4)
        if money is not None else None,
    },
    "allowed_actions": ["giải thích reason codes", "soạn thư nhắc nợ theo template",
                        "tóm tắt thay đổi so với tuần trước", "trả lời câu hỏi dựa trên context pack"],
    "forbidden_actions": ["tính hoặc sửa forecast", "tự quyết định thứ tự gọi",
                          "gửi thư khi chưa có người duyệt", "nêu con số không có trong context pack"],
    "top_invoices": worklist.head(20)[
        ["priority", "invoice_id", "name_customer", "amount", "days_past_due_at_T",
         "p_paid_30d", "reason_codes"]].to_dict(orient="records"),
}
OUT_CONTEXT.write_text(json.dumps(context_pack, ensure_ascii=False, indent=2, default=str),
                       encoding="utf-8")
print(f"Context pack: {OUT_CONTEXT.name} ({OUT_CONTEXT.stat().st_size/1024:.1f} KB)")
print(json.dumps({k: v for k, v in context_pack.items()
                  if k in ("as_of", "regime", "gate_decision", "portfolio")},
                 ensure_ascii=False, indent=2))

Context pack: agent_context_pack.json (8.3 KB)
{
  "as_of": "2020-05-22",
  "regime": "crisis",
  "gate_decision": "CONDITIONAL",
  "portfolio": {
    "open_amount_usd": 312538768.52,
    "open_invoices": 9681,
    "cash_30d_headline_usd": 95427069.56,
    "cash_30d_headline_source": "Survival B (chế độ khủng hoảng)",
    "cash_30d_alternative_usd": 307944641.97,
    "cash_30d_alternative_source": "LightGBM + CQR + Monte Carlo (giả định hồi phục)",
    "risk_amount_30d_usd": 14369659.46
  }
}


In [7]:
# ── Bản nháp thư nhắc nợ sinh bằng TEMPLATE tất định, không cần LLM ──────
TEMPLATE = """**Hoá đơn {invoice_id} — {name}**

Kính gửi Quý khách hàng {name},

Chúng tôi ghi nhận hoá đơn **{invoice_id}** trị giá **${amount:,.2f}**, đến hạn ngày
**{due}** {status}.

Theo hệ thống theo dõi công nợ của chúng tôi: {reasons}.

Mong Quý khách xác nhận kế hoạch thanh toán trước ngày **{deadline}**. Nếu đã thanh toán,
xin bỏ qua thư này và gửi giúp chúng tôi chứng từ chuyển khoản.

Trân trọng,
Phòng Công nợ

---
*Ưu tiên nội bộ #{priority} · xác suất thu trong 30 ngày: {p30:.0%} · chế độ hệ thống: {regime}*
*Thư này do hệ thống soạn tự động và **phải được người phụ trách duyệt trước khi gửi**.*
"""

drafts = []
for _, r in worklist.head(5).iterrows():
    dpd = int(r["days_past_due_at_T"])
    status = f"— đã quá hạn **{dpd} ngày**" if dpd > 0 else "— sắp đến hạn"
    drafts.append(TEMPLATE.format(
        invoice_id=r["invoice_id"], name=str(r["name_customer"]).strip(),
        amount=r["amount"], due=pd.Timestamp(r["due_date"]).date(), status=status,
        reasons=r["reason_codes"], deadline=(AS_OF + pd.Timedelta(days=7)).date(),
        priority=int(r["priority"]), p30=r["p_paid_30d"], regime=REGIME))

OUT_DRAFTS.write_text("\n\n".join(drafts), encoding="utf-8")
print(f"Bản nháp thư: {OUT_DRAFTS.name} ({len(drafts)} thư)")
print()
print(drafts[0])
print()
print("Điểm mấu chốt của D61: mọi CON SỐ trong thư đều lấy từ trường của context pack. Nếu")
print("thay template này bằng LLM, LLM chỉ đổi cách diễn đạt — nó không có đường nào để đổi số.")
print("Đó là cách biến 'không để LLM tự tính forecast' từ một lời hứa thành một ràng buộc kỹ thuật.")

Bản nháp thư: dunning_drafts.md (5 thư)

**Hoá đơn 1930645119 — SIMPLO associates**

Kính gửi Quý khách hàng SIMPLO associates,

Chúng tôi ghi nhận hoá đơn **1930645119** trị giá **$653,644.80**, đến hạn ngày
**2020-03-27** — đã quá hạn **56 ngày**.

Theo hệ thống theo dõi công nợ của chúng tôi: quá hạn 56 ngày; giá trị top 10%; ít lịch sử — độ tin cậy thấp; xác suất thu trong 30 ngày chỉ 26%.

Mong Quý khách xác nhận kế hoạch thanh toán trước ngày **2020-05-29**. Nếu đã thanh toán,
xin bỏ qua thư này và gửi giúp chúng tôi chứng từ chuyển khoản.

Trân trọng,
Phòng Công nợ

---
*Ưu tiên nội bộ #1 · xác suất thu trong 30 ngày: 26% · chế độ hệ thống: crisis*
*Thư này do hệ thống soạn tự động và **phải được người phụ trách duyệt trước khi gửi**.*


Điểm mấu chốt của D61: mọi CON SỐ trong thư đều lấy từ trường của context pack. Nếu
thay template này bằng LLM, LLM chỉ đổi cách diễn đạt — nó không có đường nào để đổi số.
Đó là cách biến 'không để LLM tự tính forecast' từ một lời hứa thành một

## 6. Xuất dashboard HTML tự chứa (D62, D63)

In [8]:
REGIME_STYLE = {
    "crisis": (C_CRIT, "CRISIS MODE", "Hành vi thanh toán đã đổi chế độ. Mọi con số dự báo "
                                      "phải đọc kèm kịch bản stress."),
    "drift_warning": (C_WARN, "CẢNH BÁO DRIFT", "Có dấu hiệu dịch chuyển. Tăng tần suất review."),
    "normal": (C_GOOD, "BÌNH THƯỜNG", "Không phát hiện dịch chuyển chế độ."),
}
rc, rlabel, rdesc = REGIME_STYLE.get(REGIME, (C_INK2, REGIME.upper(), ""))

CONF_BADGE = {"đã kiểm chứng": ("#0ca30c", "✓"), "thận trọng": ("#fab219", "!"),
              "chưa kiểm chứng": ("#d03b3b", "⚠")}


def kpi_card(row):
    color, mark = CONF_BADGE.get(row["confidence"], ("#52514e", ""))
    if row["unit"] == "USD":
        val = f"${row['value']/1e6:,.1f}M" if abs(row["value"]) >= 1e6 else f"${row['value']:,.0f}"
    elif row["unit"] == "tỷ lệ":
        val = f"{row['value']:.1%}"
    elif row["unit"] == "hoá đơn":
        val = f"{row['value']:,.0f}"
    else:
        val = f"{row['value']:,.3f}"
    return f"""<div class="card">
      <div class="card-label">{row['kpi']}</div>
      <div class="card-value">{val}</div>
      <div class="card-meta"><span class="badge" style="background:{color}">{mark} {row['confidence']}</span>
        <span class="src">{row['source']}</span></div>
    </div>"""


def df_to_html(df, cols, headers, fmts=None):
    fmts = fmts or {}
    head = "".join(f"<th>{h}</th>" for h in headers)
    rows = []
    for _, r in df.iterrows():
        tds = []
        for c in cols:
            v = r[c]
            f = fmts.get(c)
            tds.append(f"<td>{f(v) if f else v}</td>")
        rows.append("<tr>" + "".join(tds) + "</tr>")
    return f"<table><thead><tr>{head}</tr></thead><tbody>{''.join(rows)}</tbody></table>"


top_work = worklist.head(15)
work_html = df_to_html(
    top_work,
    ["priority", "invoice_id", "name_customer", "amount", "days_past_due_at_T",
     "p_paid_30d", "risk_amount_30d", "reason_codes"],
    ["#", "Hoá đơn", "Khách hàng", "Giá trị", "Quá hạn", "P(thu 30d)", "Tiền rủi ro", "Lý do"],
    {"amount": lambda v: f"${v:,.0f}", "risk_amount_30d": lambda v: f"${v:,.0f}",
     "p_paid_30d": lambda v: f"{v:.0%}", "days_past_due_at_T": lambda v: f"{int(v)}d",
     "name_customer": lambda v: str(v)[:24]})

gate_html = df_to_html(
    gate[["rule", "value", "role", "kết luận"]],
    ["rule", "value", "role", "kết luận"], ["Điều kiện", "Giá trị", "Vai trò", "Kết luận"],
    {"kết luận": lambda v: f"<span style='color:{C_GOOD if v=='PASS' else C_CRIT};font-weight:600'>{v}</span>"})

LIMITS = [
    "Con số cash-in dựa trên model được train ở giai đoạn TRƯỚC cú sốc 03/2020; trên cửa sổ "
    "hậu cú sốc, thực tế chỉ thu được vài phần trăm giá trị AR trong 14 ngày (notebook 08 §6).",
    "`clear_date` được dùng thay cho ngày tiền thực về (`bank_txn_date`) — chưa có dữ liệu ngân hàng.",
    "Ngưỡng kế hoạch là proxy từ lịch sử thu 13 tuần cuối, KHÔNG phải kế hoạch chi thật của công ty.",
    "Kịch bản stress là what-if tất định, KHÔNG kèm xác suất.",
    "Gate Tầng 9 hiện là CONDITIONAL: model thắng baseline ở mức hoá đơn nhưng chưa thắng ở mức dòng tiền.",
    "Nhóm quá hạn trên 60 ngày không đủ dữ liệu calibration riêng — dùng fallback bảo thủ.",
]

html = f"""<meta charset="utf-8">
<title>AR Cash-In Dashboard — {AS_OF.date()}</title>
<style>
  * {{ box-sizing: border-box; }}
  body {{ font-family: -apple-system, "Segoe UI", Roboto, sans-serif; margin: 0;
         background: #f9f9f7; color: #0b0b0b; line-height: 1.5; }}
  .wrap {{ max-width: 1180px; margin: 0 auto; padding: 24px 20px 60px; }}
  h1 {{ font-size: 24px; margin: 0 0 4px; }}
  h2 {{ font-size: 16px; margin: 34px 0 12px; padding-bottom: 6px;
        border-bottom: 1px solid #d8d7d2; }}
  .sub {{ color: #52514e; font-size: 13px; margin-bottom: 18px; }}
  .banner {{ background: {rc}; color: #fff; padding: 12px 16px; border-radius: 8px;
             margin: 16px 0 22px; }}
  .banner b {{ font-size: 15px; letter-spacing: .04em; }}
  .banner p {{ margin: 4px 0 0; font-size: 13px; opacity: .95; }}
  .grid {{ display: grid; grid-template-columns: repeat(auto-fit, minmax(215px, 1fr)); gap: 12px; }}
  .card {{ background: #fcfcfb; border: 1px solid #e6e5e0; border-radius: 8px; padding: 13px 15px; }}
  .card-label {{ font-size: 11px; color: #52514e; text-transform: uppercase;
                 letter-spacing: .05em; min-height: 28px; }}
  .card-value {{ font-size: 25px; font-weight: 650; margin: 3px 0 7px; }}
  .card-meta {{ font-size: 10.5px; color: #52514e; }}
  .badge {{ color: #fff; padding: 2px 7px; border-radius: 10px; font-size: 10px; margin-right: 6px; }}
  .src {{ opacity: .85; }}
  .split {{ display: grid; grid-template-columns: 1fr 1fr; gap: 14px; }}
  .panel {{ background: #fcfcfb; border: 1px solid #e6e5e0; border-radius: 8px; padding: 16px; }}
  .panel h3 {{ margin: 0 0 8px; font-size: 13px; }}
  img {{ max-width: 100%; border-radius: 6px; display: block; }}
  table {{ width: 100%; border-collapse: collapse; font-size: 12px; }}
  th {{ text-align: left; padding: 7px 8px; background: #f0efec; font-weight: 600;
        border-bottom: 1px solid #d8d7d2; }}
  td {{ padding: 6px 8px; border-bottom: 1px solid #eeede9; vertical-align: top; }}
  tbody tr:hover {{ background: #f4f4f1; }}
  .limits {{ background: #fff8e6; border: 1px solid #fab219; border-radius: 8px; padding: 16px; }}
  .limits li {{ margin-bottom: 7px; font-size: 12.5px; }}
  .foot {{ margin-top: 34px; font-size: 11px; color: #52514e; }}
  .scroll {{ overflow-x: auto; }}
  @media (max-width: 820px) {{ .split {{ grid-template-columns: 1fr; }} }}
</style>
<div class="wrap">
  <h1>Dự báo dòng tiền thu từ công nợ</h1>
  <div class="sub">Ngày dữ liệu: <b>{AS_OF.date()}</b> · Danh mục:
     {len(inv_risk):,} hoá đơn đang mở · ${open_amount:,.0f}
     · Gate Tầng 9: <b>{GATE_DECISION}</b></div>

  <div class="banner">
    <b>⬤ {rlabel}</b>
    <p>{rdesc}</p>
    <p style="margin-top:6px">Hành động bắt buộc: {regime['required_actions'].iloc[0]}</p>
  </div>

  <h2>1 · Chỉ số chính</h2>
  <div class="grid">{''.join(kpi_card(r) for _, r in kpi.iterrows())}</div>

  <h2>2 · Dòng tiền dự kiến 91 ngày</h2>
  <img src="data:image/png;base64,{img_cash}" alt="Cash-in forecast">
  <div class="panel" style="margin-top:12px">
    <h3>Vì sao có hai con số ở mốc 30 ngày?</h3>
    <p style="font-size:12.5px;margin:0">
      Hai model trả lời hai câu hỏi khác nhau. <b>Monte Carlo</b> (${lgbm_30/1e6:,.0f}M) giả định
      thị trường quay lại trạng thái 2019. <b>Survival B</b> (${surv_b_30/1e6:,.0f}M) giả định chế độ
      khủng hoảng hiện tại tiếp diễn. Chênh lệch <b>${abs(lgbm_30-surv_b_30)/1e6:,.0f}M</b> chính là
      cái giá của việc chưa biết chế độ nào sẽ đúng. Vì regime hiện tại là
      <b>{REGIME}</b>, số dùng để lập kế hoạch là nhánh thận trọng.
    </p>
  </div>

  <h2>3 · Sức chịu đựng và rủi ro tập trung</h2>
  <div class="split">
    <div><img src="data:image/png;base64,{img_stress}" alt="Stress scenarios"></div>
    <div><img src="data:image/png;base64,{img_cust}" alt="Top customers"></div>
  </div>

  <h2>4 · Danh sách thu hồi tuần tới</h2>
  <div class="panel scroll">{work_html}</div>

  <h2>5 · Kết quả gate Tầng 9</h2>
  <div class="panel scroll">{gate_html}</div>

  <h2>6 · Hạn chế đã biết</h2>
  <div class="limits">
    <ul>{''.join(f'<li>{x}</li>' for x in LIMITS)}</ul>
  </div>

  <div class="foot">
    Sinh tự động từ notebook <code>10_dashboard.ipynb</code> ·
    Nguồn: notebooks 01→09 · Mọi con số truy ngược được về artifact trong <code>data/processed/</code>.<br>
    Agent chỉ được dùng để giải thích và soạn thư; mọi con số đến từ context pack đã tính sẵn
    (<code>reports/agent_context_pack.json</code>).
  </div>
</div>
"""

OUT_HTML.write_text(html, encoding="utf-8")
print(f"Dashboard: {OUT_HTML}")
print(f"  kích thước: {OUT_HTML.stat().st_size/1024:.0f} KB (tự chứa, mở bằng trình duyệt)")

Dashboard: D:\CURRICULUM\Semester VI\Finance Project\reports\dashboard.html
  kích thước: 202 KB (tự chứa, mở bằng trình duyệt)


## 7. Lưu artifact và sanity check

In [9]:
view.to_csv(OUT_INVOICE_VIEW, index=False)
kpi.to_csv(OUT_KPI, index=False)
print(f"[1] {OUT_INVOICE_VIEW.name}  ({len(view):,} dòng)")
print(f"[2] {OUT_KPI.name}")
print(f"[3] {OUT_HTML.relative_to(BASE)}")
print(f"[4] {OUT_CONTEXT.relative_to(BASE)}")
print(f"[5] {OUT_DRAFTS.relative_to(BASE)}")

checks = []


def check(name, cond, detail=""):
    checks.append({"check": name, "pass": bool(cond), "detail": detail})


html_text = OUT_HTML.read_text(encoding="utf-8")
check("dashboard tự chứa (không có link ngoài)",
      "http://" not in html_text and "https://" not in html_text)
check("dashboard hiển thị regime", rlabel in html_text)
check("dashboard hiển thị CẢ HAI con số 30 ngày (D59)",
      f"{lgbm_30/1e6:,.0f}M" in html_text and f"{surv_b_30/1e6:,.0f}M" in html_text)
check("panel hạn chế có mặt (D63)", "Hạn chế đã biết" in html_text and len(LIMITS) >= 5)
check("mọi KPI có nhãn độ tin cậy", kpi["confidence"].isin(CONF_BADGE).all())
check("context pack có danh sách hành động bị cấm (D61)",
      len(context_pack["forbidden_actions"]) >= 4)
check("mọi con số trong thư nháp đều có trong context pack",
      all(str(r["invoice_id"]) in json.dumps(context_pack, default=str)
          for _, r in worklist.head(5).iterrows()))
check("bảng hoá đơn dashboard phủ hết pool đang mở", len(view) == len(inv_risk))

checks_df = pd.DataFrame(checks)
print()
print(checks_df.to_string(index=False))
print()
print("✅ TẦNG 11 PASS — pipeline Tầng 0→11 đã hoàn tất" if checks_df["pass"].all()
      else "❌ Có check FAIL")

[1] dashboard_invoice_view.csv  (9,681 dòng)
[2] dashboard_kpi.csv
[3] reports\dashboard.html
[4] reports\agent_context_pack.json
[5] reports\dunning_drafts.md

                                              check  pass detail
            dashboard tự chứa (không có link ngoài)  True       
                          dashboard hiển thị regime  True       
     dashboard hiển thị CẢ HAI con số 30 ngày (D59)  True       
                         panel hạn chế có mặt (D63)  True       
                         mọi KPI có nhãn độ tin cậy  True       
   context pack có danh sách hành động bị cấm (D61)  True       
mọi con số trong thư nháp đều có trong context pack  True       
        bảng hoá đơn dashboard phủ hết pool đang mở  True       

✅ TẦNG 11 PASS — pipeline Tầng 0→11 đã hoàn tất


## Tổng kết Tầng 11 — và của toàn bộ Tầng 6→11

### Đầu ra của notebook này
| File | Nội dung |
|---|---|
| `reports/dashboard.html` | Dashboard 1 file, mở bằng trình duyệt, không cần server |
| `reports/agent_context_pack.json` | Hợp đồng dữ liệu cho agent — mọi con số agent được phép nói |
| `reports/dunning_drafts.md` | 5 thư nhắc nợ nháp, sinh bằng template tất định |
| `dashboard_invoice_view.csv` | Bảng hoá đơn kèm reason code, nguồn, độ tin cậy |
| `dashboard_kpi.csv` | KPI kèm nhãn độ tin cậy |

### Ba nguyên tắc thiết kế đáng bảo vệ trước hội đồng

1. **Dashboard hiển thị sự bất đồng giữa các model, không giấu nó** (D59). Khi hai model
   chênh nhau hàng trăm triệu USD, việc chọn con số nào là quyết định quản trị rủi ro — và
   dashboard ghi rõ ai chọn, theo tiêu chí nào.

2. **Mỗi con số mang nhãn độ tin cậy** (D60). Con số nào có backtest ngoài mẫu, con số nào
   chưa — người đọc thấy ngay, không phải lật phụ lục.

3. **Ràng buộc với LLM là ràng buộc kỹ thuật, không phải lời hứa** (D61). Agent chỉ nhận
   context pack đã tính sẵn; nó không có đường nào để tạo ra một con số mới. Bản nháp thư
   được sinh bằng template để chứng minh phần sinh chữ không cần LLM để đúng số.

### Toàn cảnh Tầng 6 → 11

| Tầng | Notebook | Kết quả then chốt |
|---|---|---|
| 6 · Calibration | `04_calibration` | Coverage test 75,6% → 84,7%; chọn biến thể CQR bằng validation nội bộ |
| 7 · Monte Carlo | `05_monte_carlo` | Phát hiện dải cash-in vô dụng do giả định độc lập; sửa bằng copula hiệu chỉnh |
| 8 · Stress & change-point | `06_stress_test` | 13 kịch bản; điểm gãy chế độ **2020-03-14** ổn định qua mọi penalty; PSI 6,2 |
| 9 · Evaluation | `07_evaluation` | Gate **CONDITIONAL**: thắng ở mức hoá đơn, chưa thắng ở mức dòng tiền |
| 10A · Survival | `08_survival` | Lưới cutoff cũ **không thể** chứa hoá đơn mở; hậu cú sốc thực thu ~0,2%/14 ngày vs model ~71% |
| 10B · Collection | `09_collection` | Classifier giữ; EWMA chỉ ở mức chẩn đoán; two-part default bị loại bằng số liệu |
| 11 · Dashboard | `10_dashboard` | Dashboard tự chứa + hợp đồng agent + worklist có reason code |

### Việc còn lại nếu đi tiếp Vòng 2
1. Lấy `bank_txn_date` hoặc bản trích xuất muộn hơn để phân biệt cú sốc thật với độ trễ ghi nhận.
2. Train lại toàn bộ trục chính trên lưới cutoff mở rộng, không chỉ tầng survival.
3. Sinh feature survival bằng out-of-fold trước khi đưa Survival-Boost vào production.
4. Thay ngưỡng kế hoạch proxy bằng kế hoạch chi thật của công ty.